# 02 - new domains, frozen encoders on the prepared datasets (GPU)

Pre-registered 2026-09-26 (`src/sartransfer/domains/run.py`, nothing tuned per
domain). DINOv3 ViT-L SAT-493M at layer 21 and C-RADIOv4-H at layer 32 (layers
from the glacier study); SAR input = (co-pol, cross-pol, co - cross)
standardised on the training split; a linear probe and the glacier study's
row-3 decoder (3 seeds); every non-train split scored in the same run, since
nothing is chosen on any of them.

**Runtime** - A100 with High-RAM. Run all cells in order. Cells 1-3 set up, 4 checks the
alpine values (read only), 5 runs every dataset x encoder pair (each encoder is loaded
once) and saves each pair's results to Drive (`MyDrive/sar-transfer-part2/results/`);
a pair already there is skipped, so after a crash just run cell 5 again. Cell 6 prints
the table. Measured 2026-09-29, forest 3-4 min per encoder, all four datasets with both encoders about 1 h 35 min.
Secrets and model weights come from part one's Drive folder (`MyDrive/sar-transfer/.env`,
`MyDrive/sar-transfer/hf_cache`).

In [1]:
# --- cell - mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# <<<SARTRANSFER-SYNC-BEGIN>>>
# Empty sync cell: published notebooks carry no packed code. On your own machine,
# in the repository folder, run   python sync.py   and open the notebook again;
# this cell then holds the package source and unpacks it on the runtime.
raise RuntimeError("The sync cell is empty. On your own machine, in the repository folder, "
                   "run:  python sync.py  and reopen this notebook.")
# <<<SARTRANSFER-SYNC-END>>>

In [ ]:
# --- cell - secrets, GPU, RAM, prepared data to local disk ---
import os, shutil, time
from pathlib import Path
import psutil, torch
from sartransfer.env import load_env, require, model_cache_on_drive

load_env(keys=("HF_TOKEN",))                       # the read token only; the write token never enters this runtime
require("HF_TOKEN", "gated model weights")
model_cache_on_drive()
assert torch.cuda.is_available(), "switch to a GPU runtime"
g = torch.cuda.get_device_properties(0)
print(f"gpu: {g.name} | GPU memory {g.total_memory / 1e9:.0f} GB | system RAM {psutil.virtual_memory().total / 1e9:.0f} GB")
DRIVE = Path("/content/drive/MyDrive/sar-transfer-part2")  # part two: converted data + results;
                                                          # .env and hf_cache stay in MyDrive/sar-transfer
RES = DRIVE / "results"
PREP = Path("/content/domains_prepared")           # local disk: a few large files per dataset
from sartransfer.domains import hub
# 1st your private Hugging Face copy (fast), used only if it is the same conversion as the one on
# Drive and HF_TOKEN can read it; else the Drive copy. Only complete copies are accepted.
DATASETS = sorted(hub.stage(("glacial_lakes", "snow", "forty_v1", "glavitu_alp"), PREP, DRIVE / "domains",
                            owner="saverin00", token=os.environ["HF_TOKEN"]))
print("prepared datasets:", DATASETS, "| free disk", f"{shutil.disk_usage('/content').free / 1e9:.0f} GB")

In [4]:
# --- cell - CHECK -- alpine pixels above 20 dB, per chip (read only, under a minute) ---
# The converter printed test values up to 169.8 dB co-pol / 112 dB cross-pol (train max 14.5 dB).
# The encoder gets the standardised values unclipped, so this shows how many pixels and which chips.
import numpy as np
from sartransfer.domains import prepared as P
for s in ("train", "val", "test"):
    d = P.load_split(PREP, "glavitu_alp", s)
    x = np.asarray(d["images"], np.float32)                    # (chips, 2, 992, 992) dB, NaN outside the tile
    n_hi = (x > 20).sum(axis=(2, 3))                           # pixels above 20 dB, per chip and channel
    n_ok = np.isfinite(x).sum(axis=(2, 3))
    bad = np.flatnonzero(n_hi.any(1))
    print(f"{s}: {len(x)} chips, {len(bad)} with pixels above 20 dB (train max is 14.5 dB)")
    for i in bad:
        m = d["meta"].iloc[i]
        print(f"  chip {i} tile {m['tile']} {m['orbit']}: co {n_hi[i, 0]:,} of {n_ok[i, 0]:,} px "
              f"(max {np.nanmax(x[i, 0]):.1f} dB), cross {n_hi[i, 1]:,} of {n_ok[i, 1]:,} px (max {np.nanmax(x[i, 1]):.1f} dB)")
    del x

train: 177 chips, 0 with pixels above 20 dB (train max is 14.5 dB)
val: 59 chips, 0 with pixels above 20 dB (train max is 14.5 dB)
test: 60 chips, 1 with pixels above 20 dB (train max is 14.5 dB)
  chip 42 tile ALP-31-33 desc: co 4 of 867,180 px (max 169.8 dB), cross 6 of 867,180 px (max 112.1 dB)


In [5]:
# --- cell - RUN -- every prepared dataset x both encoders (skips pairs already on Drive) ---
import gc, torch
from sartransfer.domains.run import run_domain

for encoder in ("dinov3-l-sat", "cradio-v4-h"):
    enc_cache = {}                                   # load each encoder once, for all datasets
    for dataset in DATASETS:
        t0 = time.perf_counter()
        try:
            run_domain(PREP, dataset, encoder, RES, token=os.environ["HF_TOKEN"], enc_cache=enc_cache)
        except Exception as e:
            print(f"FAILED {dataset} {encoder}: {type(e).__name__}: {str(e)[:300]}")
        print(f"{dataset} x {encoder}: {(time.perf_counter() - t0) / 60:.0f} min\n")
    enc_cache.clear(); gc.collect(); torch.cuda.empty_cache()

skip dom_forty_v1__dinov3-l-sat__L21__vv_vh_diff: summary already has probe + decoder seeds [0, 1, 2]
forty_v1 x dinov3-l-sat: 0 min

  NOTE: SAR units are 'minmax01_per_chip', not dB; results are reported with that label

===== dom_glacial_lakes__dinov3-l-sat__L21__vv_vh_diff: splits {'test': 1834, 'test_challenge': 1105, 'test_region_SEE': 613, 'train': 14820, 'val': 1851} | classes ['background', 'lake'] | chip 256 px =====


Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

dinov3-l-sat: 303.1 M parameters loaded
  encoder loaded in 0.1 min
  features test: (1834, 16, 16, 1024) in 0.1 min
  features test_challenge: (1105, 16, 16, 1024) in 0.1 min
  features test_region_SEE: (613, 16, 16, 1024) in 0.0 min
  features train: (14820, 16, 16, 1024) in 1.0 min
  features val: (1851, 16, 16, 1024) in 0.1 min
  test: 790 labelled pixels ignored for missing SAR
  test_challenge: 147 labelled pixels ignored for missing SAR
  test_region_SEE: 0 labelled pixels ignored for missing SAR
  train: 5,981 labelled pixels ignored for missing SAR
  val: 752 labelled pixels ignored for missing SAR
  probe: 2,000,000 patches, 33 epochs x 31 steps, 0.3 min
    training tiles held on cuda (2.6 GB)
    step  1000  loss 0.3720  0.2 min
    step  2000  loss 0.2942  0.3 min
    step  3000  loss 0.2563  0.5 min
    step  4000  loss 0.2374  0.7 min
  decoder seed 0: 0.8 min
    training tiles held on cuda (2.6 GB)
    step  1000  loss 0.3647  0.2 min
    step  2000  loss 0.2955  0.3 m

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

remote code verified: 26 files == reviewed copy (nvidia/C-RADIOv4-H@0057b339059c)


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

cradio-v4-h: 651.6 M parameters loaded
  encoder loaded in 0.1 min
  features test: (1834, 16, 16, 1280) in 0.1 min
  features test_challenge: (1105, 16, 16, 1280) in 0.1 min
  features test_region_SEE: (613, 16, 16, 1280) in 0.0 min
  features train: (14820, 16, 16, 1280) in 1.2 min
  features val: (1851, 16, 16, 1280) in 0.2 min
  test: 790 labelled pixels ignored for missing SAR
  test_challenge: 147 labelled pixels ignored for missing SAR
  test_region_SEE: 0 labelled pixels ignored for missing SAR
  train: 5,981 labelled pixels ignored for missing SAR
  val: 752 labelled pixels ignored for missing SAR
  probe: 2,000,000 patches, 33 epochs x 31 steps, 0.3 min
    training tiles held on cuda (3.1 GB)
    step  1000  loss 0.3983  0.2 min
    step  2000  loss 0.3258  0.3 min
    step  3000  loss 0.3035  0.5 min
    step  4000  loss 0.2901  0.7 min
  decoder seed 0: 0.8 min
    training tiles held on cuda (3.1 GB)
    step  1000  loss 0.3893  0.2 min
    step  2000  loss 0.3493  0.3 mi

In [6]:
# --- cell - TABLE -- every dataset, encoder, head and split (decoder: mean +- SD over 3 seeds) ---
import pandas as pd
from sartransfer.domains.run import domain_table

pd.set_option("display.width", 220, "display.precision", 3)
print(domain_table(RES).to_string(index=False))

      dataset      encoder     head           split  seeds  mIoU   mIoU_sd  accuracy  IoU_pos  IoU_pos_sd
     forty_v1  cradio-v4-h  decoder            test      3 0.350 8.435e-03     0.533      NaN         NaN
     forty_v1  cradio-v4-h  decoder             val      3 0.344 1.024e-02     0.533      NaN         NaN
     forty_v1  cradio-v4-h majority            test      1 0.040       NaN     0.319      NaN         NaN
     forty_v1  cradio-v4-h majority             val      1 0.039       NaN     0.313      NaN         NaN
     forty_v1  cradio-v4-h    probe            test      1 0.391       NaN     0.575      NaN         NaN
     forty_v1  cradio-v4-h    probe             val      1 0.374       NaN     0.563      NaN         NaN
     forty_v1 dinov3-l-sat  decoder            test      3 0.384 9.373e-03     0.581      NaN         NaN
     forty_v1 dinov3-l-sat  decoder             val      3 0.377 1.341e-02     0.573      NaN         NaN
     forty_v1 dinov3-l-sat majority           